In [ ]:
#workshopping Historical_FWI.py generalisation 

#first need to recreate the iberia work but more efficient to check data processing is working.



In [ ]:
#cylc stuff 

[task parameters]
    
    member = 1..15

[scheduling]

    [[graph]]
        P1Y = """
            hadgem_historical_fwi<member>
        """

[runtime]
    [[hadgem_historical_fwi<member>]]
        script = """
        set -eux
        conda activate sowf
        cd /data/users/bob.potts/StateOfFires_2025-26/code
        python hadgem_historical_fwi.py
        """

        platform = spice

        [[[directives]]]
            --mem = 100G
            --time = 60
            --cpus-per-task = 1
            --ntasks = 1

In [5]:
import iris
import numpy as np
from utils.constrain_cubes_standard import *
from utils.cubefuncs import *
import time

In [ ]:
#CONFIG

country = 'Iberia'
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############


folder = '/data/scratch/chantelle.burton/SoW2526/'

#Set up the 2025 files and months automatically
if country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')
      

elif country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')




In [ ]:
#CONFIG

country = 'Iberia'
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############


folder = '/data/scratch/chantelle.burton/SoW2526/'

#Set up the 2025 files and months automatically
if country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')
      

elif country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    daterange = iris.Constraint(time=lambda cell: cell.point.month == Month)
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

start_time = time.time()
ERA5_ImpactsToolBox_Arr = []
for year in np.arange(1960, 2014):
    print('ERA5',year)
    ERA5_ImpactsToolBox = iris.load_cube(folder+'/historicalFWI/ERA5/FWI_era5_era5_era5_'+str(year)+'0'+str(Month)+'01*.nc', 'canadian_fire_weather_index')
    ERA5_ImpactsToolBox = TimePercentile(ERA5_ImpactsToolBox, percentile)
    ERA5_ImpactsToolBox = contrain_to_sow_shapefile(ERA5_ImpactsToolBox, '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp', 'Northwest Iberia')
    ERA5_ImpactsToolBox = CountryPercentile(ERA5_ImpactsToolBox, percentile)
    ERA5_ImpactsToolBox = np.ravel(ERA5_ImpactsToolBox.data)
    ERA5_ImpactsToolBox_Arr.append(ERA5_ImpactsToolBox)
    print(ERA5_ImpactsToolBox)

#Save ERA5 out to a text file
f = open(folder+'/output/ERA5_FWI_1960-2013_'+country+str(percentile)+'%.dat','a')
np.savetxt(f,(ERA5_ImpactsToolBox_Arr))
f.close()    
print('finished')
print("--- %s seconds ---" % (np.round(time.time() - start_time, 2)))

KeyboardInterrupt: 

In [26]:
import iris
import numpy as np
from utils.constrain_cubes_standard import *
from utils.cubefuncs import *
import time
import glob
import iris.coord_categorisation as icc
import warnings
import re
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
# CONFIG
country = 'Iberia'
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)

START_YEAR = 1960
END_YEAR = 2013

folder = '/data/scratch/chantelle.burton/SoW2526/'
shp_file = '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp'

# Set up the 2025 files and months automatically
if country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    shape_name = 'South Korea'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    shape_name = 'Northwest Iberia'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    shape_name = 'Scotland'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

start_time = time.time()

# Load all historical files once (for selected month and year range)
hist_pattern = folder + f'/historicalFWI/ERA5/FWI_era5_era5_era5_*{Month:02d}01-*.nc'
all_files = sorted(glob.glob(hist_pattern))

# Filter files by year range
hist_files = []
pattern = re.compile(rf'_(\d{{4}}){Month:02d}01-\d{{8}}_')
for f in all_files:
    match = pattern.search(f)
    if match:
        year = int(match.group(1))
        if START_YEAR <= year <= END_YEAR:
            hist_files.append(f)

print(f"Found {len(hist_files)} files for years {START_YEAR}-{END_YEAR}")

if not hist_files:
    raise FileNotFoundError(f"No historical ERA5 files found in year range {START_YEAR}-{END_YEAR}")

print(f"Loading {len(hist_files)} files...")
# Use concatenate instead of merge
cubes = iris.load(hist_files, 'canadian_fire_weather_index')
for cube in cubes:
    for coord_name in ("year", "season_year"):
        if cube.coords(coord_name):
            cube.remove_coord(coord_name)

ERA5_hist_all = cubes.concatenate_cube()

# Cut to shapefile once
print("Applying shapefile")
ERA5_hist_all = contrain_to_sow_shapefile(ERA5_hist_all, shp_file, shape_name)

# Add year coordinate
try:
    icc.add_year(ERA5_hist_all, 'time')
except ValueError:
    pass  # already exists

# 1) Percentile over time within each year
print("Computing time percentile by year...")
yr_time_p = ERA5_hist_all.aggregated_by('year', iris.analysis.PERCENTILE, percent=percentile)

# 2) Percentile over space (lat/lon) for each year
print("Computing spatial percentile for each year...")
yr_country_p = yr_time_p.collapsed(['latitude', 'longitude'], iris.analysis.PERCENTILE, percent=percentile)

# Final 1D array by year
ERA5_ImpactsToolBox_Arr = np.ravel(yr_country_p.data)

# Save ERA5 out to a text file
output_file = f'/data/scratch/bob.potts/sowf/test_output/IRIS_ERA5_FWI_{START_YEAR}-{END_YEAR}_'+country+str(percentile)+'%.dat'
np.savetxt(output_file, ERA5_ImpactsToolBox_Arr)

print('Finished')
print(f"Data shape: {ERA5_ImpactsToolBox_Arr.shape}")
print(f"Saved to: {output_file}")
print("--- %s seconds ---" % (np.round(time.time() - start_time, 2)))

Running Iberia
Found 54 files for years 1960-2013
Loading 54 files...
Applying shapefile
Computing time percentile by year...
Computing spatial percentile for each year...
Finished
Data shape: (54,)
Saved to: /data/scratch/bob.potts/sowf/test_output/IRIS_ERA5_FWI_1960-2013_Iberia95%.dat
--- 72.14 seconds ---


In [30]:
# ...existing code...
import numpy as np

file_a = "/data/scratch/chantelle.burton/SoW2526/output/ERA5_FWI_1960-2013_Iberia95%.dat"
file_b = "/data/scratch/bob.potts/sowf/test_output/IRIS_ERA5_FWI_1960-2013_Iberia95%.dat"

a = np.loadtxt(file_a)
b = np.loadtxt(file_b)

print("len a:", len(a), "len b:", len(b))
print("first 5 a:", a[:5])
print("first 5 b:", b[:5])
import numpy as np

print("len a:", len(a), "len b:", len(b))
n = min(len(a), len(b))
print("max abs diff:", np.max(np.abs(a[:n]-b[:n])))
print("mean abs diff:", np.mean(np.abs(a[:n]-b[:n])))


len a: 54 len b: 54
first 5 a: [29.70566406 46.59042969 46.44121094 40.10722656 39.90292969]
first 5 b: [29.70566406 46.59042969 46.44121094 40.10722656 39.90292969]
len a: 54 len b: 54
max abs diff: 0.0
mean abs diff: 0.0


In [31]:
import geopandas as gpd
gdf = gpd.read_file('/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp')
print(gdf.columns)
print(gdf['name'].unique()) 

Index(['UID', 'name', 'geometry'], dtype='str')
<StringArray>
[                      'Northwest Iberia',
     'Midwestern Canadian Shield forests',
 'Chilean Temperate Forests and Matorral',
                     'Scottish Highlands',
                  'Southeast South Korea']
Length: 5, dtype: str
